# 77 — Generate the overlay SHACL shapes

The `55_` analogue for the overlay: `overlay/qbc.schema.yaml` through `gen-shacl`,
written to `cosmos_qbc_v1.shapes.ttl` at repo root. Runs after
`10_fetch_cosmos.ipynb` and `20_generate.ipynb` (the schema imports the patched BC
model) and after `70_generate_qbc.ipynb` (the enum repair below resolves against the
core T-Box, the same way `70_` does).

Unlike the two core shapes graphs, which are generated and shipped unmodified
because they are CDISC's constraints, this one carries **one authored edit, named
rather than silent — decision D24.** `gen-shacl` writes every enum constraint as
`sh:in` over string literals, while the A-Box carries the permissible values as the
IRIs `gen-owl` declares (D20). For the core rendering that disagreement is recorded
and left standing (`60_validate_instances.ipynb`, decision D11). For the overlay's
own shapes it is repaired: the three `sh:in` lists are rewritten to the IRIs the
A-Box actually carries. And for the two result-scale properties the list is not
CDISC's five values but the three the overlay admits (D23) — which is the closed-world
half of D23 stated declaratively, in the deliverable, rather than only in a renderer
guard and a CI check.

Decisions exercised: **D6** (every generator option pinned), **D9** (canonicalize
before writing), **D19** (the shape for the SDTM stand-in is named
`cosmos_sdtm:AssignedTerm`, because `gen-shacl` honours `class_uri`), **D20**, **D23**,
**D24**.

## Configuration

In [ ]:
ROOT    = ".."
OVERLAY = "../overlay"

SCHEMA = f"{OVERLAY}/qbc.schema.yaml"
SCALES = f"{OVERLAY}/scales.instances.yaml"
TARGET = "cosmos_qbc_v1.shapes.ttl"

QBC_TBOX  = f"{ROOT}/cosmos_qbc_v1.ttl"
CORE_TBOX = f"{ROOT}/cosmos_bc_v1.ttl"

QBC_NS = "https://w3id.org/cdisc/cosmos/qbc/"
BC_NS  = "https://www.cdisc.org/cosmos/biomedical_concept_v1.0/"

## Generate

`exclude_imports` is the one option that matters, and it is the shapes-side twin of
`mergeimports: False` in `70_`. Left at its default, `gen-shacl` emits node shapes for
`cosmos_bc:BiomedicalConcept`, `cosmos_bc:DataElementConcept` and `cosmos_bc:Coding`
into the overlay's shapes graph — a second copy of CDISC's constraints, under this
repo's file, which is what decision D7 declined. Measured 2026-09-04: 12 node shapes
with imports, 9 without. Every other option is left at the same default `55_` uses,
stated here so a generator upgrade changes nothing silently (D6).

In [ ]:
from pathlib import Path

from linkml.generators.shaclgen import ShaclGenerator
from rdflib import Graph, URIRef
from rdflib.namespace import RDF, SH

# Pinned to the linkml 1.11.1 defaults 55_generate_shapes.ipynb runs with, except
# exclude_imports (decision D6).
SHACL_OPTIONS = {
    "exclude_imports": True,
    "closed": True,
    "include_annotations": False,
    "use_class_uri_names": True,
    "expand_subproperty_of": True,
}

graph = Graph().parse(data=ShaclGenerator(SCHEMA, **SHACL_OPTIONS).serialize(), format="turtle")

targets = sorted(str(o) for o in graph.objects(None, SH.targetClass))
print(f"{TARGET:28s} {len(graph):>5,} triples generated  <- {SCHEMA}")
print(f"{'':28s} {len(targets)} node shapes")
for target in targets:
    print(f"{'':28s}   {target}")

## Repair the enum constraints — decision D24

`gen-shacl` writes an enum constraint as `sh:in` over the permissible values' text.
The A-Box does not carry text: a result scale is the core enum's IRI (D20), a mapping
relation is the overlay enum's IRI. Validated as generated, every such value fails
`sh:in` — 41 results at this pin, all the same generator disagreement `60_` records
for the core under D11.

Here the disagreement is repaired rather than recorded, because these are the
overlay's own shapes and the overlay's own reading of them. Each `sh:in` list is
rewritten to IRIs resolved out of the T-Box that declares the enum — the same guard
`75_render_qbc.ipynb` uses, so a value the T-Box does not have raises. For
`resultScale` and `permissibleValue` the list is **the admitted set of D23**, read
from `overlay/scales.instances.yaml` — three values, not five — and
`sh:nodeKind sh:IRI` is added alongside. The guard asserts that exactly the three
expected lists exist before the repair, and that no string-valued `sh:in` survives
after it.

In [ ]:
import yaml
from rdflib import BNode
from rdflib.collection import Collection

core_tbox = Graph().parse(CORE_TBOX, format="turtle")
qbc_tbox = Graph().parse(QBC_TBOX, format="turtle")


def core_enum(enum_name, value):
    iri = URIRef(f"{BC_NS}{enum_name}#{value}")
    if (iri, RDF.type, OWL_CLASS) not in core_tbox:
        raise RuntimeError(f"{value!r} is not a permissible value of core {enum_name}")
    return iri


def qbc_enum(enum_name, value):
    iri = URIRef(f"{QBC_NS}{enum_name}#{value}")
    if (iri, RDF.type, OWL_CLASS) not in qbc_tbox:
        raise RuntimeError(f"{value!r} is not a permissible value of overlay {enum_name}")
    return iri


OWL_CLASS = URIRef("http://www.w3.org/2002/07/owl#Class")

admitted = [s["permissibleValue"] for s in yaml.safe_load(Path(SCALES).read_text(encoding="utf-8"))["resultScales"]]

# property -> the IRIs its sh:in list must hold
REPAIR = {
    "resultScale":     [core_enum("BiomedicalConceptResultScaleEnum", v) for v in admitted],
    "permissibleValue": [core_enum("BiomedicalConceptResultScaleEnum", v) for v in admitted],
    "relation":        [qbc_enum("MappingRelationEnum", v) for v in ("exactMatch", "narrowMatch", "broadMatch")],
}

found = {}
for shape, lst in graph.subject_objects(SH["in"]):
    prop = str(graph.value(shape, SH.path)).replace(QBC_NS, "")
    found[prop] = (shape, lst)
if set(found) != set(REPAIR):
    raise RuntimeError(f"sh:in lists changed: found {sorted(found)}, expected {sorted(REPAIR)}")

for prop, iris in REPAIR.items():
    shape, lst = found[prop]
    before = [str(v) for v in Collection(graph, lst)]
    Collection(graph, lst).clear()
    graph.remove((shape, SH["in"], lst))
    head = BNode()
    Collection(graph, head, iris)
    graph.add((shape, SH["in"], head))
    graph.add((shape, SH.nodeKind, SH.IRI))
    print(f"qbc:{prop}")
    print(f"    was  {before}")
    for iri in iris:
        print(f"    now  {iri}")

remaining = [str(v) for _, lst in graph.subject_objects(SH["in"]) for v in Collection(graph, lst) if not isinstance(v, URIRef)]
if remaining:
    raise RuntimeError(f"string-valued sh:in members survive the repair: {remaining}")
print()
print(f"{TARGET:28s} {len(graph):>5,} triples after repair")

## Canonicalize and write — decision D9

Same procedure as `55_`: shapes graphs are mostly blank nodes, so the canonical form
is what makes the file byte-stable across runs and platforms.

In [ ]:
from rdflib.compare import isomorphic, to_canonical_graph


def canonicalize(source):
    canonical = to_canonical_graph(source)
    if not isomorphic(canonical, source) or len(canonical) != len(source):
        raise RuntimeError("canonicalization changed the graph")
    result = Graph()
    for triple in canonical:
        result.add(triple)
    result.bind("sh", SH)
    result.bind("qbc", QBC_NS)
    result.bind("cosmos_bc", BC_NS)
    result.bind("cosmos_sdtm", "https://www.cdisc.org/cosmos/sdtm_v1.0/")
    return result


canonical = canonicalize(graph)
turtle = canonical.serialize(format="turtle")
again = canonicalize(Graph().parse(data=turtle, format="turtle")).serialize(format="turtle")
if again != turtle:
    raise RuntimeError(f"{TARGET}: canonical serialization is not stable")

Path(ROOT, TARGET).write_text(turtle, encoding="utf-8")
print(f"{TARGET:28s} {len(canonical):>5,} triples  {len(turtle):>7,} chars  stable")

## Confirm what was generated

1. Nine node shapes, all closed, targeting the overlay's own classes — and, per D19,
   the stand-in's shape is named `cosmos_sdtm:AssignedTerm` because `gen-shacl`
   honours `class_uri` where `gen-owl` does not. No shape targets a class of the
   imported BC model.
2. The two result-scale `sh:in` lists hold exactly the admitted IRIs of D23, and the
   two values the core enum carries beyond them are absent.

In [ ]:
written = Graph().parse(Path(ROOT, TARGET), format="turtle")

shapes = set(written.subjects(RDF.type, SH.NodeShape))
closed = {s for s in shapes if (s, SH.closed, None) in written}
target_classes = {str(o) for o in written.objects(None, SH.targetClass)}
print(f"sh:NodeShape     {len(shapes):>3}   ({len(closed)} closed)")
print(f"property shapes  {len(list(written.subject_objects(SH.path))):>3}")
if len(shapes) != 9 or closed != shapes:
    raise RuntimeError(f"expected 9 closed node shapes, found {len(shapes)} ({len(closed)} closed)")
if any(t.startswith(BC_NS) for t in target_classes):
    raise RuntimeError("a shape targets a class of the imported BC model")

scale_enum = f"{BC_NS}BiomedicalConceptResultScaleEnum#"
for prop in ("resultScale", "permissibleValue"):
    shape = next(s for s in written.subjects(SH.path, URIRef(QBC_NS + prop)))
    members = sorted(str(v).replace(scale_enum, "") for v in Collection(written, written.value(shape, SH["in"])))
    if members != sorted(admitted):
        raise RuntimeError(f"qbc:{prop} sh:in is {members}, expected {sorted(admitted)}")
    print(f"qbc:{prop:17s} sh:in {members}")
core_values = sorted(str(v).replace(scale_enum, "") for v in core_tbox.subjects(RDF.type, OWL_CLASS) if str(v).startswith(scale_enum))
print(f"core enum carries      {core_values}; left out by D23: {sorted(set(core_values) - set(admitted))}")

## What this notebook does not produce

The JSON-LD context for the overlay — the `40_` analogue — is still not generated.
The conformance run of the overlay A-Box against these shapes is
`78_validate_qbc_instances.ipynb`.